In [0]:
%sql
--BRONZE
DROP TABLE IF EXISTS workspace.yelp_bronze.fraud_rule;
DROP TABLE IF EXISTS workspace.yelp_bronze.trust_safety_area;

-- bronze_regla_fraude
CREATE TABLE IF NOT EXISTS workspace.yelp_bronze.fraud_rule (
  fraud_rule_id          STRING,
  fraud_description         STRING,
  risk_level        STRING,
  rule_status       STRING,
  ingestion_date       STRING,
  source_file      STRING
);

-- bronze_area_trust_safety
CREATE TABLE IF NOT EXISTS workspace.yelp_bronze.trust_safety_area (
  team_id        STRING,
  team_name    STRING,
  responsible      STRING,
  team_status    STRING,
  ingestion_date    STRING,
  source_file   STRING
);

In [0]:
%sql
-- ============================================================
-- SILVER LAYER DDL
-- Schema: workspace.yelp_silver
-- Source model: Hackolade Delta Lake export + bronze raw tables
-- 7 tables: user, business, review, checkin, tip, fraud_rule, trust_safety_area
-- Quality rules: NOT NULL constraints on PKs, FKs, and key business columns
-- Relationships:
--   user.user_id              → review.user_id          (FK)
--   business.business_id      → review.business_id      (FK)
--   business.business_id      → checkin.business_id     (FK)
--   business.business_id      → tip.business_id         (FK)
--   user.user_id              → tip.user_id             (FK)

-- ============================================================

-- Create the silver schema inside the existing 'workspace' catalog
CREATE SCHEMA IF NOT EXISTS workspace.yelp_silver;

-- Drop existing silver tables (idempotent re-run)
DROP TABLE IF EXISTS workspace.yelp_silver.fraud_rule;
DROP TABLE IF EXISTS workspace.yelp_silver.trust_safety_area;
DROP TABLE IF EXISTS workspace.yelp_silver.tip;
DROP TABLE IF EXISTS workspace.yelp_silver.checkin;
DROP TABLE IF EXISTS workspace.yelp_silver.review;
DROP TABLE IF EXISTS workspace.yelp_silver.business;
DROP TABLE IF EXISTS workspace.yelp_silver.user;

-- ----------------------------------------------------------
-- user
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.user (
  user_id           BIGINT       NOT NULL,
  user_name         STRING       NOT NULL,
  email             STRING,
  average_stars     DOUBLE       NOT NULL,
  review_count      BIGINT       NOT NULL,
  fans              BIGINT       NOT NULL,
  elite             STRING,
  friends           STRING,
  yelping_since     TIMESTAMP    NOT NULL,
  useful            BIGINT       NOT NULL,
  funny             BIGINT       NOT NULL,
  cool              BIGINT       NOT NULL,
  account_status    STRING,
  load_date         TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_user PRIMARY KEY (user_id)
);

-- ----------------------------------------------------------
-- business
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.business (
  business_id       BIGINT       NOT NULL,
  business_name     STRING       NOT NULL,
  category          STRING       NOT NULL,
  city              STRING       NOT NULL,
  state             STRING       NOT NULL,
  latitude          DOUBLE,
  longitude         DOUBLE,
  stars             DOUBLE       NOT NULL,
  review_count      BIGINT       NOT NULL,
  is_open           BOOLEAN      NOT NULL,
  business_status   STRING,
  load_date         TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_business PRIMARY KEY (business_id)
);

-- ----------------------------------------------------------
-- review  (FK → user, business)
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.review (
  review_id          BIGINT       NOT NULL,
  user_id            BIGINT       NOT NULL,
  business_id        BIGINT       NOT NULL,
  rating             SMALLINT     NOT NULL,
  comment            STRING       NOT NULL,
  useful             BIGINT       NOT NULL,
  funny              BIGINT       NOT NULL,
  cool               BIGINT       NOT NULL,
  published_date     TIMESTAMP    NOT NULL,
  review_status      STRING,
  load_date          TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_review PRIMARY KEY (review_id),
  CONSTRAINT fk_review_user FOREIGN KEY (user_id)
    REFERENCES workspace.yelp_silver.user (user_id),
  CONSTRAINT fk_review_business FOREIGN KEY (business_id)
    REFERENCES workspace.yelp_silver.business (business_id)
);

-- ----------------------------------------------------------
-- checkin  (FK → business)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.checkin (
  checkin_id         BIGINT       NOT NULL,
  business_id        BIGINT       NOT NULL,
  checkin_date       TIMESTAMP    NOT NULL,
  load_date          TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_checkin PRIMARY KEY (checkin_id),
  CONSTRAINT fk_checkin_business FOREIGN KEY (business_id)
    REFERENCES workspace.yelp_silver.business (business_id)
);

-- ----------------------------------------------------------
-- tip  (FK → business, user)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.tip (
  tip_id            BIGINT       NOT NULL,
  business_id       BIGINT       NOT NULL,
  user_id           BIGINT       NOT NULL,
  tip_text          STRING       NOT NULL,
  compliment_count  BIGINT       NOT NULL,
  tip_date          TIMESTAMP    NOT NULL,
  load_date         TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_tip PRIMARY KEY (tip_id),
  CONSTRAINT fk_tip_business FOREIGN KEY (business_id)
    REFERENCES workspace.yelp_silver.business (business_id),
  CONSTRAINT fk_tip_user FOREIGN KEY (user_id)
    REFERENCES workspace.yelp_silver.user (user_id)
);

-- ----------------------------------------------------------
-- trust_safety_area
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.trust_safety_area (
  team_id           BIGINT       NOT NULL,
  team_name         STRING       NOT NULL,
  responsible       STRING       NOT NULL,
  team_status       STRING       NOT NULL,
  load_date         TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_trust_safety PRIMARY KEY (team_id)
);

-- ----------------------------------------------------------
-- fraud_rule
CREATE TABLE IF NOT EXISTS workspace.yelp_silver.fraud_rule (
  fraud_rule_id     BIGINT       NOT NULL,
  fraud_description STRING       NOT NULL,
  risk_level        STRING       NOT NULL,
  rule_status       STRING       NOT NULL,
  load_date         TIMESTAMP    NOT NULL,
  CONSTRAINT pk_silver_fraud PRIMARY KEY (fraud_rule_id)
);

In [0]:
%sql
--yelp_gold
DROP TABLE IF EXISTS workspace.yelp_gold.review_assessment;
DROP TABLE IF EXISTS workspace.yelp_gold.review_assessment_criteria;

CREATE TABLE IF NOT EXISTS workspace.yelp_gold.review_assessment (
  review_id          BIGINT       NOT NULL,
  user_id            BIGINT       NOT NULL,
  business_id        BIGINT       NOT NULL,
  assessment_result  STRING       NOT NULL,
  assessment_score   SMALLINT     NOT NULL,
  assessment_date   TIMESTAMP     NOT NULL,
  load_date          TIMESTAMP    NOT NULL,
  CONSTRAINT pk_review_id PRIMARY KEY (review_id)
);

CREATE TABLE IF NOT EXISTS workspace.yelp_gold.review_assessment_criteria (
  criteria_id        BIGINT       NOT NULL,
  review_id          BIGINT       NOT NULL,
  result_flag        SMALLINT     NOT NULL,
  assessment_date   TIMESTAMP     NOT NULL,
  load_date          TIMESTAMP    NOT NULL,
  CONSTRAINT pk_criteria_id PRIMARY KEY (criteria_id, review_id),
  CONSTRAINT fk_review_id FOREIGN KEY (review_id)
    REFERENCES workspace.yelp_gold.review_assessment (review_id)
);